#### Notebook to extract details of individual researchers' oeuvres 

In [24]:
%run common_setup.ipynb

#### Match works to authors  

- Construct time-series of citations from reference lists  
- Compute the centile for each publication year
- Filter highly cited papers  
- Group the authors of hte highly cited papers

In [26]:
class CorpusETL(SetUp):

    def __init__(self):
        super().__init__()
        return

    def citations_per_work(self):
    # -- SQL FOR citations_per_work(self):
    # -- =================================
        sql = """ 
            CREATE OR REPLACE TABLE memory.citations_per_work AS
                SELECT cited_id,
                        count(citer_id) AS cited_by_count_endogenous,
                        cited_by_count AS cited_by_count_total,
                        publication_year
                FROM project.citer_cited
                JOIN project.raw 
                ON cited_id = id
                GROUP BY ALL 
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.citations_per_work ORDER BY cited_by_count_total DESC, cited_by_count_endogenous DESC").show()
        return

    def citations_per_work_ranked(self):
    # -- SQL FOR def citations_per_work_ranked(self):
    # -- ============================================
        sql = """ 
            CREATE OR REPLACE TABLE memory.citations_per_work_ranked AS
                SELECT cited_id,
                        publication_year,
                        cited_by_count_total,
                        cited_by_count_endogenous,
                        percent_rank(ORDER BY cited_by_count_total) OVER w AS percent_rank_total,
                        percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_endogenous
                FROM memory.citations_per_work 
                WINDOW w AS (PARTITION BY publication_year) 
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.citations_per_work_ranked ORDER BY publication_year DESC, percent_rank_endogenous DESC").show()
        return

    def author_citations(self):       
    # -- SQL FOR def author_citations(self):
    # -- ============================================
        sql = """ 
            CREATE OR REPLACE TABLE memory.author_citations AS
            WITH
            -- unique works-authors
            author_works_CTE AS
                (SELECT DISTINCT work_id,
                        author_id,
                        author_name,
                    FROM project.authorships
                    WHERE author_id NOT NULL
                ),
            -- order by ciaton_count and then rank (from memory.citations_per_work)
            ranker_CTE AS
                (SELECT cited_id,
                        publication_year,
                        cited_by_count_total,
                        cited_by_count_endogenous,
                        percent_rank(ORDER BY cited_by_count_total) OVER w AS percent_rank_total,
                        percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_endogenous,
                FROM memory.citations_per_work 
                WINDOW w AS (PARTITION BY publication_year)
                ORDER BY publication_year DESC, percent_rank_endogenous DESC
                ),
            -- assemble all the cited works to each author
            author_citations_CTE AS
                (SELECT DISTINCT author_id,
                        author_name,
                        sum(cited_by_count_total) OVER (PARTITION BY author_id) AS author_cited_by_count_total,
                        sum(cited_by_count_endogenous) OVER (PARTITION BY author_id) AS author_cited_by_count_endogenous,
                        percent_rank_total,
                        percent_rank_endogenous,
                FROM ranker_CTE r
                JOIN author_works_CTE a
                ON work_id = cited_id
                -- GROUP BY ALL
                ORDER BY author_cited_by_count_endogenous DESC
                ),
            -- count the HCA total
            total_hca_CTE AS
                (SELECT DISTINCT author_id,
                                    author_name,
                                    count() AS author_hca_count_total
                            FROM author_citations_CTE
                            WHERE percent_rank_total >= 0.99
                            GROUP BY ALL
                ),
            -- count the HCA endogenous
            endogenous_hca_CTE AS
                (SELECT DISTINCT author_id,
                                    author_name,
                                    
                                    count() AS author_hca_count_endogenous
                            FROM author_citations_CTE
                            WHERE percent_rank_endogenous >= 0.99
                            GROUP BY ALL
                )

            -- assemble author-level citation caounts
            SELECT DISTINCT ON (a.author_id)
                    a.author_id,
                    a.author_name,
                    author_cited_by_count_total,
                    author_cited_by_count_endogenous,
                    author_hca_count_total,
                    author_hca_count_endogenous
            FROM author_citations_CTE a
            LEFT JOIN total_hca_CTE
            USING (author_id, author_name)
            LEFT JOIN endogenous_hca_CTE 
            USING (author_id, author_name)
            ORDER BY author_hca_count_endogenous DESC
        """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.author_citations ORDER BY author_hca_count_endogenous DESC").show()
        return
    
    def author_works_count(self):
    # -- SQL FOR def author_works_count(self):
    # -- =====================================
        sql = """
            CREATE OR REPLACE TABLE memory.author_works_counts AS
                SELECT DISTINCT au.author_id,
                        au.author_name,
                        -- a.first,
                        -- a.middle,
                        -- a.last,
                        -- a.fullname,
                        a.orcid,
                        -- a.display_name_alternatives,
                        count(work_id) AS works_count_endogenous,
                        works_count AS works_count_total,
                        cited_by_count AS cited_by_count_total,
                        h_index,
                        h_index_2yr
                    FROM project.authorships au
                        LEFT JOIN project.authors a
                        ON a.author_id = au.author_id
                    WHERE au.author_id NOT NULL
                    GROUP BY ALL
                    -- ORDER BY works_count_endogenous DESC
                """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.author_works_counts ORDER BY works_count_endogenous DESC").show()
        return
        
    def citation_summary(self):
    # -- SQL FOR def citation_summary(self):
    # -- ===================================
        sql = """ 
            CREATE OR REPLACE TABLE project.citation_summary AS
                SELECT DISTINCT 
                        author_id,
                        author_name,
                        author_cited_by_count_total,
                        author_cited_by_count_endogenous,
                        author_hca_count_total,
                        author_hca_count_endogenous,
                        orcid,
                        works_count_endogenous,
                        works_count_total,
                        cited_by_count_total,
                        h_index,
                        h_index_2yr
                FROM memory.author_citations a
                LEFT JOIN memory.author_works_counts
                USING (author_id, author_name)
            ORDER BY author_cited_by_count_endogenous DESC
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.citation_summary ORDER BY works_count_endogenous DESC").show()
        return
    
    def load_citations(self):
        df = self.db.sql("SELECT * FROM project.citation_summary ORDER BY works_count_endogenous DESC").df().reset_index(drop=True)
        df.to_excel('../DATA/citation_summary.xlsx', index=False)
        return


In [27]:
def main():

    so = SampleOeuvres()
    so.db.close()

    cetl = CorpusETL()
    cetl.citations_per_work()
    cetl.citations_per_work_ranked()
    cetl.author_citations()
    cetl.author_works_count()
    cetl.citation_summary()
    cetl.load_citations()
    cetl.db.close()
        
    return

In [28]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────┬───────────┐
│   database   │ schema  │         name         │     column_names     │           column_types            │ temporary │
│   varchar    │ varchar │       varchar        │      varchar[]       │             varchar[]             │  boolean  │
├──────────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────┼───────────┤
│ authors      │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ institutions │ main    │ institutions         │ [id, ror, display_…  │ [VARCHAR, VARCHAR, VARCHAR, VAR…  │ false     │
│ institutions │ main    │ ror                  │ [name, institution…  │ [VARCHAR, VARCHAR]                │ false     │
│ project      │ main    │ author_citation_re…  │ [dupes, author_id,…  │ [BIGINT, VARCHAR, VARCHAR, VARC…  │ false     │
│ project      │ main    │ autho